In [1]:
import pandas as pd
import json
import pickle
import os
from rank_bm25 import BM25Okapi
from tqdm import tqdm

print("Libraries imported!")

Libraries imported!


In [2]:
# Load chunks
chunks_df = pd.read_csv('data/chunks_512.csv')

# Load test questions
with open('data/test_questions.json', 'r') as f:
    test_questions = json.load(f)

print(f"Loaded {len(chunks_df)} chunks")
print(f"Loaded {len(test_questions)} test questions")

Loaded 3725 chunks
Loaded 20 test questions


In [3]:
print("Building BM25 index...")

# Tokenize all chunks
corpus = chunks_df['text'].tolist()
tokenized_corpus = [doc.lower().split() for doc in tqdm(corpus)]

# Build BM25
bm25 = BM25Okapi(tokenized_corpus)

print(f"BM25 index built on {len(corpus)} chunks!")

Building BM25 index...


100%|██████████| 3725/3725 [00:00<00:00, 25835.32it/s]


BM25 index built on 3725 chunks!


In [4]:
os.makedirs('src/retrievers', exist_ok=True)

# Save BM25 index
with open('src/retrievers/bm25_index.pkl', 'wb') as f:
    pickle.dump(bm25, f)

# Save corpus separately
with open('src/retrievers/corpus.pkl', 'wb') as f:
    pickle.dump(corpus, f)

print("BM25 index saved!")

BM25 index saved!


In [5]:
def bm25_retrieve(query, top_k=5):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    
    # Get top k indices
    top_indices = scores.argsort()[-top_k:][::-1]
    
    results = []
    for idx in top_indices:
        results.append({
            'chunk_id': chunks_df.iloc[idx]['chunk_id'],
            'text': corpus[idx],
            'score': scores[idx]
        })
    
    return results

print("Retriever function ready!")

Retriever function ready!


In [6]:
print("=== TESTING BM25 RETRIEVER ===\n")

for i, item in enumerate(test_questions[:5]):
    query = item['question']
    results = bm25_retrieve(query, top_k=3)
    
    print(f"Question {i+1}: {query}")
    print(f"Top result score: {results[0]['score']:.4f}")
    print(f"Top result preview: {results[0]['text'][:150]}...")
    print("-" * 50)

=== TESTING BM25 RETRIEVER ===

Question 1: What type of system is being analyzed in the paper for the mean resolvent using a polymer expansion?
Top result score: 47.7675
Top result preview: in this paper we develop a polymer expansion with large / small field conditions for the mean resolvent of a weakly disordered system . then we show t...
--------------------------------------------------
Question 2: What is the significance of the asymptotic expansion for the density of states in the context of the research paper?
Top result score: 55.5793
Top result preview: in this paper we develop a polymer expansion with large / small field conditions for the mean resolvent of a weakly disordered system . then we show t...
--------------------------------------------------
Question 3: What is the asymptotic long-time equivalence being referred to in the context of the paper?
Top result score: 39.0474
Top result preview: we discuss the non - relativistic limit of quantum field theory in an iner

In [7]:
print("Evaluating BM25 on all test questions...\n")

bm25_results = []

for item in tqdm(test_questions):
    query = item['question']
    results = bm25_retrieve(query, top_k=5)
    
    # Check if source abstract appears in top 5 results
    source = item['source_abstract']
    retrieved_texts = [r['text'] for r in results]
    
    # Hit = source abstract words found in retrieved chunks
    hit = any(
        len(set(source.lower().split()) & 
            set(r.lower().split())) > 20 
        for r in retrieved_texts
    )
    
    bm25_results.append({
        'question': query,
        'hit': hit,
        'top_score': results[0]['score'],
        'top_result': results[0]['text']
    })

# Calculate accuracy
hits = sum(1 for r in bm25_results if r['hit'])
accuracy = hits / len(bm25_results) * 100

print(f"BM25 Results:")
print(f"Total questions: {len(bm25_results)}")
print(f"Correct retrievals: {hits}")
print(f"Accuracy: {accuracy:.1f}%")

Evaluating BM25 on all test questions...



100%|██████████| 20/20 [00:00<00:00, 23.76it/s]

BM25 Results:
Total questions: 20
Correct retrievals: 20
Accuracy: 100.0%


In [8]:
os.makedirs('results', exist_ok=True)

# Save detailed results
with open('results/bm25_results.json', 'w') as f:
    json.dump(bm25_results, f, indent=2)

# Save summary
summary = {
    'retriever': 'BM25',
    'total_questions': len(bm25_results),
    'hits': hits,
    'accuracy': accuracy
}

with open('results/bm25_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Results saved!")
print(f"\nBM25 Summary:")
print(f"Accuracy: {accuracy:.1f}%")

Results saved!

BM25 Summary:
Accuracy: 100.0%
